In [8]:
pip install xlrd
pip install pandas openpyxl

   ---------------------------------------- 0.0/96.5 kB ? eta -:--:--
   ---- ----------------------------------- 10.2/96.5 kB ? eta -:--:--
   ---------------------------------------- 96.5/96.5 kB 1.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [114]:
import tkinter as tk
from tkinter import filedialog
import pandas as pd
import os

class ExcelFileOpener:
    def __init__(self, root):
        self.root = root
        self.root.title("Excel File Opener")
        self.selected_dataframes = {}  # Dictionary to hold DataFrames for selected sheets
        self.file_name = None  # Variable to hold the concatenated values of the first two columns

        # Create a button to open the file dialog
        self.open_button = tk.Button(root, text="Open Excel File", command=self.open_file)
        self.open_button.pack(pady=20)

    def open_file(self):
        # Open a file dialog to select an Excel file
        file_path = filedialog.askopenfilename(
            title="Select an Excel file",
            filetypes=(("Excel files", "*.xls;*.xlsx"), ("All files", "*.*"))
        )
        if file_path:
            try:
                # Read all sheets from the Excel file into a dictionary of DataFrames
                all_sheets = pd.read_excel(file_path, sheet_name=None)

                # Initialize summary DataFrame
                summary_data = []

                # Iterate through all sheets
                for sheet_name, data in all_sheets.items():
                    # Select rows 14 to 24 (0-indexed: 13 to 23)
                    selected_rows = data.iloc[13:24]  # Select rows 14 to 24

                    # Drop rows where all elements are NaN
                    cleaned_rows = selected_rows.dropna(how='all')

                    # Store all values in the first two columns
                    first_col_values = cleaned_rows.iloc[:, 0].fillna('')  # Get column 1 and fill na with empty strings
                    
                    # Create a valid file name from the concatenated values and drop the first 9 characters
                    valid_file_name = ''.join(first_col_values).strip()[9:]  # Join the first column values, Drop the first 9 characters and strip spaces

                    # Check if the valid_file_name is empty and set to "unknown_title"
                    if not valid_file_name:
                        valid_file_name = "unknown_title"

                    # Truncate valid_file_name to a maximum of 31 characters
                    valid_file_name = valid_file_name[:31]

                    # Replace any spaces in the valid file name with underscores
                    valid_file_name = valid_file_name.replace(" ", "_")

                    # Drop the first seven columns
                    remaining_cols = cleaned_rows.drop(columns=cleaned_rows.columns[:7], errors='ignore')

                    # Clean remaining columns
                    remaining_cleaned = remaining_cols.apply(lambda row: row.dropna().reset_index(drop=True), axis=1)

                    # Store the cleaned DataFrame in the dictionary with a valid file name
                    self.selected_dataframes[valid_file_name] = remaining_cleaned.reset_index(drop=True)

                # Create the summary DataFrame
                summary_data = {}
                unique_values = set()

                # Collect unique values from column 0 (excluding the first row) and their corresponding values from column 3
                for df_name, df in self.selected_dataframes.items():
                    if df.shape[1] > 3:  # Ensure there are at least 4 columns
                        col_0_values = df[0].unique()[1:]  # Skip the first row
                        unique_values.update(col_0_values)

                        for value in col_0_values:
                            if value not in summary_data:
                                summary_data[value] = {df_name: []}
                            # Append all corresponding values from column 3, excluding the first row
                            col_3_values = df.loc[df[0] == value, 3].tolist()
                            summary_data[value][df_name] = col_3_values if col_3_values else [0]  # Use 0 if no values

                # Create the summary DataFrame
                summary_df = pd.DataFrame(index=list(unique_values))  # Convert set to list

                # Fill the summary DataFrame with values from each DataFrame
                for df_name in self.selected_dataframes.keys():
                    summary_df[df_name] = summary_df.index.map(
                        lambda x: summary_data.get(x, {}).get(df_name, 0)  # Change to 0 instead of [0]
                    )

                # Ensure that all values are numeric, converting lists to single values
                for df_name in summary_df.columns:
                    summary_df[df_name] = summary_df[df_name].apply(lambda x: x[0] if isinstance(x, list) else x)

                # Save all DataFrames and summary to a new Excel file
                output_file_path = os.path.splitext(file_path)[0] + "_output.xlsx"  # Create output file name
                with pd.ExcelWriter(output_file_path) as writer:
                    # Write all DataFrames to separate sheets
                    for df_name, df in self.selected_dataframes.items():
                        df.to_excel(writer, sheet_name=df_name, index=False)

                    # Write the summary DataFrame to a sheet named "Summary"
                    summary_df.to_excel(writer, sheet_name=f"Summary", index=True)

                print(f"\nDataFrames and summary saved to: {output_file_path}")

                # After saving the summary DataFrame
                with pd.ExcelWriter(output_file_path, mode='a') as writer:  # Open in append mode
                    # Create a DataFrame for the analysis results
                    analysis_data = {
                        'Component': ['GLY'],
                        'Initial Glycerol Yield': [initial_glycerol] if initial_glycerol is not None else [0]
                    }
                    analysis_df = pd.DataFrame(analysis_data)
                
                    # Save the analysis DataFrame to a new sheet
                    analysis_df.to_excel(writer, sheet_name="Analysis", index=False)
                
                print(f"\nAnalysis DataFrame saved to: {output_file_path}")

                # Close the GUI after opening the file
                self.root.destroy()

            except Exception as e:
                print(f"Error opening file: {e}")

# Create the main window
root = tk.Tk()
app = ExcelFileOpener(root)

# Start the GUI event loop
root.mainloop()


WARNING *** file size (61976) not 512 + multiple of sector size (512)

DataFrames and summary saved to: C:/Users/smarlow/OneDrive - Imperial College London/Sushila Marlow/1 Biomass Oxidation Optimisation/1 HPLC Data/24.10.03_CA_Degrade Test 2/24-10-03_HPLC_Short_Results_GOR_CA_Trial1_output.xlsx


In [11]:
import tkinter as tk
from tkinter import filedialog, simpledialog
import pandas as pd
import os

class ExcelFileOpener:
    def __init__(self, root):
        self.root = root
        self.root.title("Excel File Opener")
        self.selected_dataframes = {}
        self.total_charge = None

        # Create a button to open the file dialog
        self.open_button = tk.Button(root, text="Open Excel File", command=self.open_file)
        self.open_button.pack(pady=20)

    def open_file(self):
        # Open a file dialog to select an Excel file
        file_path = filedialog.askopenfilename(
            title="Select an Excel file",
            filetypes=(("Excel files", "*.xls;*.xlsx"), ("All files", "*.*"))
        )
        if file_path:
            try:
                # Read all sheets from the Excel file into a dictionary of DataFrames
                all_sheets = pd.read_excel(file_path, sheet_name=None)

                # Initialize summary DataFrame
                summary_data = []

                # Iterate through all sheets
                for sheet_name, data in all_sheets.items():
                    selected_rows = data.iloc[13:24]
                    cleaned_rows = selected_rows.dropna(how='all')

                    # Get the file name from the first column
                    file_name = cleaned_rows.iloc[:, 0].fillna('').str.cat(sep=' ')
                    valid_file_name = file_name.replace(" ", "_")[9:]

                    if not valid_file_name:
                        valid_file_name = "unknown_title"

                    # Drop the first seven columns and clean the DataFrame
                    remaining_cols = cleaned_rows.drop(columns=cleaned_rows.columns[:7], errors='ignore')
                    if not remaining_cols.empty:
                        remaining_cols.iloc[:, 0] = remaining_cols.iloc[:, 0].shift(-2)
                        remaining_cols = remaining_cols.iloc[:, 1:]

                    remaining_cleaned = remaining_cols.apply(lambda row: row.dropna().reset_index(drop=True), axis=1)
                    self.selected_dataframes[valid_file_name] = remaining_cleaned.reset_index(drop=True)

                    # Compile first and fourth rows for summary
                    if not remaining_cleaned.empty:
                        first_row = remaining_cleaned.iloc[[0]]
                        fourth_row = remaining_cleaned.iloc[[3]] if len(remaining_cleaned) > 3 else pd.DataFrame()

                        if not first_row.empty or not fourth_row.empty:
                            combined = pd.concat([first_row, fourth_row], ignore_index=True)
                            combined.insert(0, 'File Name', valid_file_name)
                            summary_data.append(combined)

                # Create the summary DataFrame
                if summary_data:
                    summary_df = pd.concat(summary_data, ignore_index=True)

                    # Save summary DataFrame to Excel
                    summary_output_path = os.path.splitext(file_path)[0] + "_summary_output.xlsx"
                    summary_df.to_excel(summary_output_path, index=False)
                    print(f"\nSummary DataFrame saved to: {summary_output_path}")

                print("DataFrames created from selected sheets:")
                for df_name in self.selected_dataframes:
                    print(f"- {df_name}")

                # Save all DataFrames to a new Excel file
                output_file_path = os.path.splitext(file_path)[0] + "_output.xlsx"
                with pd.ExcelWriter(output_file_path) as writer:
                    for df_name, df in self.selected_dataframes.items():
                        # Rename sheet name to remove unwanted characters
                        sheet_name = df_name.split('.')[0]  # Remove any additional parts from the name
                        df.to_excel(writer, sheet_name=sheet_name[:31], index=False)

                    # Ask for total charge passed
                    self.total_charge = simpledialog.askfloat("Input", "Enter the total charge passed:")
                    if self.total_charge is not None:
                        # Calculate analysis metrics
                        self.calculate_analysis(writer)

                print(f"\nDataFrames saved to: {output_file_path}")

                # Close the GUI after opening the file
                self.root.destroy()

            except Exception as e:
                print(f"Error opening file: {e}")

    def calculate_analysis(self, writer):
        # Yield analysis
        yield_data = []
        conversion_data = []
        faradaic_data = []

        for df_name, df in self.selected_dataframes.items():
            glycerol_row = df[df.iloc[:, 0] == 'GLY']
            if not glycerol_row.empty:
                initial_glycerol = pd.to_numeric(glycerol_row.iloc[0, 3], errors='coerce')
                measured_glycerol = pd.to_numeric(df.iloc[:, 3], errors='coerce')
                consumed_glycerol = initial_glycerol - measured_glycerol

                # Yield calculation
                yield_values = (consumed_glycerol / initial_glycerol * 100).fillna(0)

                # Prepare yield analysis DataFrame
                yield_df = pd.DataFrame({
                    'Sample Name': [df_name] * len(yield_values),
                    'Initial Glycerol': initial_glycerol,
                    'Consumed Glycerol': consumed_glycerol.tolist(),
                })

                for i in range(len(yield_values)):
                    yield_df[f'Yield Product {i+1} (%)'] = yield_values[i]

                yield_data.append(yield_df)

                # Conversion analysis
                conversion_percentage = (consumed_glycerol / initial_glycerol * 100) if initial_glycerol else 0
                conversion_df = pd.DataFrame({
                    'Sample Name': df_name,
                    'Initial Glycerol': initial_glycerol,
                    'Consumed Glycerol': consumed_glycerol,
                    'Glycerol Conversion (%)': conversion_percentage
                })
                conversion_data.append(conversion_df)

                # Faradaic efficiency
                total_charge_passed = self.total_charge
                product_columns = df.columns[4:]  # Assuming products start from the 5th column
                faradaic_efficiencies = {}

                for product in product_columns:
                    product_formed = pd.to_numeric(df[product], errors='coerce')
                    faradaic_efficiency = (product_formed * 2 * 96485 / total_charge_passed) * 100
                    faradaic_efficiencies[product] = faradaic_efficiency

                faradaic_df = pd.DataFrame(faradaic_efficiencies)
                faradaic_df.insert(0, 'Sample Name', df_name)
                faradaic_data.append(faradaic_df)

        # Save all data to the writer
        if yield_data:
            final_yield_analysis = pd.concat(yield_data, ignore_index=True)
            final_yield_analysis.to_excel(writer, sheet_name="Yield Analysis", index=False)

        if conversion_data:
            final_conversion_analysis = pd.concat(conversion_data, ignore_index=True)
            final_conversion_analysis.to_excel(writer, sheet_name="Conversion Analysis", index=False)

        if faradaic_data:
            final_faradaic_efficiency = pd.concat(faradaic_data, ignore_index=True)
            final_faradaic_efficiency.to_excel(writer, sheet_name="Faradaic Efficiency", index=False)

# Create the main window
root = tk.Tk()
app = ExcelFileOpener(root)

# Start the GUI event loop
root.mainloop()


WARNING *** file size (61976) not 512 + multiple of sector size (512)

Summary DataFrame saved to: C:/Users/smarlow/OneDrive - Imperial College London/Sushila Marlow/1 Biomass Oxidation Optimisation/1 HPLC Data/24.10.03_CA_Degrade Test 2/24-10-03_HPLC_Short_Results_GOR_CA_Trial1_summary_output.xlsx
DataFrames created from selected sheets:
- _001-P1-F1-H2O_STD.D
- unknown_title
- _003-P1-F3-GLY_STD.D
- _004-P1-F4-A1.D___
- _005-P1-F5-A2.D_______
- _006-P1-F6-A3.D_______
- _007-P1-F7-C1.D______
- _008-P1-F8-C2.D___
- _009-P1-F9-C3.D____

DataFrames saved to: C:/Users/smarlow/OneDrive - Imperial College London/Sushila Marlow/1 Biomass Oxidation Optimisation/1 HPLC Data/24.10.03_CA_Degrade Test 2/24-10-03_HPLC_Short_Results_GOR_CA_Trial1_output.xlsx


In [12]:
import tkinter as tk
from tkinter import filedialog, simpledialog
import pandas as pd
import os

class ExcelFileOpener:
    def __init__(self, root):
        self.root = root
        self.root.title("Excel File Opener")
        self.selected_dataframes = {}
        self.total_charge = None

        # Create a button to open the file dialog
        self.open_button = tk.Button(root, text="Open Excel File", command=self.open_file)
        self.open_button.pack(pady=20)

    def open_file(self):
        # Open a file dialog to select an Excel file
        file_path = filedialog.askopenfilename(
            title="Select an Excel file",
            filetypes=(("Excel files", "*.xls;*.xlsx"), ("All files", "*.*"))
        )
        if file_path:
            try:
                # Read all sheets from the Excel file into a dictionary of DataFrames
                all_sheets = pd.read_excel(file_path, sheet_name=None)

                # Initialize summary DataFrame
                summary_data = []

                # Iterate through all sheets
                for sheet_name, data in all_sheets.items():
                    selected_rows = data.iloc[13:24]
                    cleaned_rows = selected_rows.dropna(how='all')

                    # Get the file name from the first column
                    file_name = cleaned_rows.iloc[:, 0].fillna('').str.cat(sep=' ')
                    valid_file_name = file_name.replace(" ", "_")[9:]

                    if not valid_file_name:
                        valid_file_name = "unknown_title"

                    # Drop the first seven columns and clean the DataFrame
                    remaining_cols = cleaned_rows.drop(columns=cleaned_rows.columns[:7], errors='ignore')
                    if not remaining_cols.empty:
                        remaining_cols.iloc[:, 0] = remaining_cols.iloc[:, 0].shift(-2)
                        remaining_cols = remaining_cols.iloc[:, 1:]

                    remaining_cleaned = remaining_cols.apply(lambda row: row.dropna().reset_index(drop=True), axis=1)
                    self.selected_dataframes[valid_file_name] = remaining_cleaned.reset_index(drop=True)

                    # Compile first and fourth rows for summary
                    if not remaining_cleaned.empty:
                        first_row = remaining_cleaned.iloc[[0]]
                        fourth_row = remaining_cleaned.iloc[[3]] if len(remaining_cleaned) > 3 else pd.DataFrame()

                        if not first_row.empty or not fourth_row.empty:
                            combined = pd.concat([first_row, fourth_row], ignore_index=True)
                            combined.insert(0, 'File Name', valid_file_name)
                            summary_data.append(combined)

                # Create the summary DataFrame
                if summary_data:
                    summary_df = pd.concat(summary_data, ignore_index=True)

                    # Save summary DataFrame to Excel
                    summary_output_path = os.path.splitext(file_path)[0] + "_summary_output.xlsx"
                    summary_df.to_excel(summary_output_path, index=False)
                    print(f"\nSummary DataFrame saved to: {summary_output_path}")

                print("DataFrames created from selected sheets:")
                for df_name in self.selected_dataframes:
                    print(f"- {df_name}")

                # Save all DataFrames to a new Excel file
                output_file_path = os.path.splitext(file_path)[0] + "_output.xlsx"
                with pd.ExcelWriter(output_file_path) as writer:
                    # Save the summary DataFrame
                    summary_df.to_excel(writer, sheet_name="Summary", index=False)

                    for df_name, df in self.selected_dataframes.items():
                        # Rename sheet name to remove unwanted characters
                        sheet_name = df_name.split('.')[0]  # Remove any additional parts from the name
                        df.to_excel(writer, sheet_name=sheet_name[:31], index=False)

                    # Ask for total charge passed
                    self.total_charge = simpledialog.askfloat("Input", "Enter the total charge passed:")
                    if self.total_charge is not None:
                        # Calculate analysis metrics
                        self.calculate_analysis(writer)

                print(f"\nDataFrames saved to: {output_file_path}")

                # Close the GUI after opening the file
                self.root.destroy()

            except Exception as e:
                print(f"Error opening file: {e}")

    def calculate_analysis(self, writer):
        # Yield analysis
        yield_data = []
        conversion_data = []
        faradaic_data = []

        for df_name, df in self.selected_dataframes.items():
            glycerol_row = df[df.iloc[:, 0] == 'GLY']
            if not glycerol_row.empty:
                initial_glycerol = pd.to_numeric(glycerol_row.iloc[0, 3], errors='coerce')
                measured_glycerol = pd.to_numeric(df.iloc[:, 3], errors='coerce')
                consumed_glycerol = initial_glycerol - measured_glycerol

                # Yield calculation
                yield_values = (consumed_glycerol / initial_glycerol * 100).fillna(0)

                # Prepare yield analysis DataFrame
                yield_df = pd.DataFrame({
                    'Sample Name': [df_name] * len(yield_values),
                    'Initial Glycerol': initial_glycerol,
                    'Consumed Glycerol': consumed_glycerol.tolist(),
                })

                for i in range(len(yield_values)):
                    yield_df[f'Yield Product {i+1} (%)'] = yield_values[i]

                yield_data.append(yield_df)

                # Conversion analysis
                conversion_percentage = (consumed_glycerol / initial_glycerol * 100) if initial_glycerol else 0
                conversion_df = pd.DataFrame({
                    'Sample Name': df_name,
                    'Initial Glycerol': initial_glycerol,
                    'Consumed Glycerol': consumed_glycerol,
                    'Glycerol Conversion (%)': conversion_percentage
                })
                conversion_data.append(conversion_df)

                # Faradaic efficiency
                total_charge_passed = self.total_charge
                product_columns = df.columns[4:]  # Assuming products start from the 5th column
                faradaic_efficiencies = {}

                for product in product_columns:
                    product_formed = pd.to_numeric(df[product], errors='coerce')
                    faradaic_efficiency = (product_formed * 2 * 96485 / total_charge_passed) * 100
                    faradaic_efficiencies[product] = faradaic_efficiency

                faradaic_df = pd.DataFrame(faradaic_efficiencies)
                faradaic_df.insert(0, 'Sample Name', df_name)
                faradaic_data.append(faradaic_df)

        # Save all data to the writer
        if yield_data:
            final_yield_analysis = pd.concat(yield_data, ignore_index=True)
            final_yield_analysis.to_excel(writer, sheet_name="Yield Analysis", index=False)

        if conversion_data:
            final_conversion_analysis = pd.concat(conversion_data, ignore_index=True)
            final_conversion_analysis.to_excel(writer, sheet_name="Conversion Analysis", index=False)

        if faradaic_data:
            final_faradaic_efficiency = pd.concat(faradaic_data, ignore_index=True)
            final_faradaic_efficiency.to_excel(writer, sheet_name="Faradaic Efficiency", index=False)

# Create the main window
root = tk.Tk()
app = ExcelFileOpener(root)

# Start the GUI event loop
root.mainloop()


WARNING *** file size (61976) not 512 + multiple of sector size (512)

Summary DataFrame saved to: C:/Users/smarlow/OneDrive - Imperial College London/Sushila Marlow/1 Biomass Oxidation Optimisation/1 HPLC Data/24.10.03_CA_Degrade Test 2/24-10-03_HPLC_Short_Results_GOR_CA_Trial1_summary_output.xlsx
DataFrames created from selected sheets:
- _001-P1-F1-H2O_STD.D
- unknown_title
- _003-P1-F3-GLY_STD.D
- _004-P1-F4-A1.D___
- _005-P1-F5-A2.D_______
- _006-P1-F6-A3.D_______
- _007-P1-F7-C1.D______
- _008-P1-F8-C2.D___
- _009-P1-F9-C3.D____

DataFrames saved to: C:/Users/smarlow/OneDrive - Imperial College London/Sushila Marlow/1 Biomass Oxidation Optimisation/1 HPLC Data/24.10.03_CA_Degrade Test 2/24-10-03_HPLC_Short_Results_GOR_CA_Trial1_output.xlsx
